In [1]:
"""
Created on Sun Aug  3 13:07:20 2025

@author: huzefa
"""
#Importing necessasary libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.svm import LinearSVC, SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import average_precision_score, recall_score, make_scorer

In [2]:
#Importing dataset
diabetes_data=pd.read_excel('../Datasets/data_file.xlsx')
diabetes_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2703 entries, 0 to 2702
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   GENDER    2703 non-null   int64  
 1   AGE       2703 non-null   int64  
 2   Height    2703 non-null   int64  
 3   Weight    2703 non-null   float64
 4   BMI       2703 non-null   float64
 5   BAI       2703 non-null   float64
 6   HBA1C1    2703 non-null   float64
 7   OGTT1FBS  2703 non-null   int64  
 8   NDD       2703 non-null   int64  
dtypes: float64(4), int64(5)
memory usage: 190.2 KB


In [3]:
#Creting a copy of original df to work upon
clean_data=diabetes_data.copy()

In [4]:
#Cleaning clean_data by pruning duplicates if present
clean_data=clean_data.drop_duplicates(keep='first')
clean_data.info()
print(clean_data.head())

<class 'pandas.core.frame.DataFrame'>
Index: 1065 entries, 0 to 2250
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   GENDER    1065 non-null   int64  
 1   AGE       1065 non-null   int64  
 2   Height    1065 non-null   int64  
 3   Weight    1065 non-null   float64
 4   BMI       1065 non-null   float64
 5   BAI       1065 non-null   float64
 6   HBA1C1    1065 non-null   float64
 7   OGTT1FBS  1065 non-null   int64  
 8   NDD       1065 non-null   int64  
dtypes: float64(4), int64(5)
memory usage: 83.2 KB
   GENDER  AGE  Height  Weight        BMI    BAI  HBA1C1  OGTT1FBS  NDD
0       1   37     156    88.0  36.160421  41.53     5.1       102    0
1       0   35     146    56.0  26.271345  34.15     5.0        91    0
2       1   54     160    76.0  29.687500  28.45     5.4        73    0
3       0   46     154    64.0  26.986001  33.28     6.0       151    0
4       0   70     156    55.0  22.600263  21.52     5.6   

In [5]:
#Assigning lables with appropriate numerics
nondia=0
diabetic=1
Ynew=pd.DataFrame(nondia,index=clean_data.index,columns=['Diabetic'])

In [6]:
#Identifying the diabetic status of each record using blood test result of FBS or HBA1C1
Ynew.iloc[list(np.where((clean_data.OGTT1FBS>=126) | (clean_data.HBA1C1>=6.5))[0])]=diabetic

In [7]:
#Concatenating diabetic status with anthroprometric features of dataset
data_df=pd.concat([clean_data.iloc[:,:6],Ynew],axis=1)
print(data_df.head())

   GENDER  AGE  Height  Weight        BMI    BAI  Diabetic
0       1   37     156    88.0  36.160421  41.53         0
1       0   35     146    56.0  26.271345  34.15         0
2       1   54     160    76.0  29.687500  28.45         0
3       0   46     154    64.0  26.986001  33.28         1
4       0   70     156    55.0  22.600263  21.52         1


In [8]:
#Finding the diabetic and non diabetic patients
diabetic_yes=data_df.iloc[list(np.where(data_df.Diabetic==diabetic)[0])]
diabetic_no=data_df.iloc[list(np.where(data_df.Diabetic==nondia)[0])]

In [9]:
#Getting basic stats about diabetic people
print(diabetic_yes.describe())

           GENDER         AGE      Height      Weight         BMI         BAI  \
count  528.000000  528.000000  528.000000  528.000000  528.000000  528.000000   
mean     0.571970   51.812500  160.280303   68.404356   26.640045   29.618864   
std      0.495262   10.921528    7.562330   12.759664    4.809005    7.753363   
min      0.000000   25.000000  138.000000   36.500000   15.390454    8.300000   
25%      0.000000   43.750000  156.000000   59.000000   23.290154   24.790000   
50%      1.000000   50.000000  159.500000   68.000000   26.527004   28.145000   
75%      1.000000   59.000000  165.000000   76.000000   29.585799   33.847500   
max      1.000000   80.000000  186.000000  102.000000   41.207076   58.250000   

       Diabetic  
count     528.0  
mean        1.0  
std         0.0  
min         1.0  
25%         1.0  
50%         1.0  
75%         1.0  
max         1.0  


In [10]:
#Getting basic stats about non diabetic people
print(diabetic_no.describe())

           GENDER         AGE      Height      Weight         BMI         BAI  \
count  537.000000  537.000000  537.000000  537.000000  537.000000  537.000000   
mean     0.450652   44.463687  158.217877   68.325885   27.261189   31.687058   
std      0.498023   11.890164    7.930377   13.118822    4.770046    8.276824   
min      0.000000   20.000000  139.000000   33.500000   13.671875   14.080000   
25%      0.000000   35.000000  153.000000   61.000000   24.508946   25.650000   
50%      0.000000   44.000000  158.000000   68.000000   26.959840   30.140000   
75%      1.000000   52.000000  162.000000   76.000000   29.757785   38.260000   
max      1.000000   84.000000  186.000000  110.000000   50.219138   59.760000   

       Diabetic  
count     537.0  
mean        0.0  
std         0.0  
min         0.0  
25%         0.0  
50%         0.0  
75%         0.0  
max         0.0  


In [11]:
#Encoding categorical variables to numeric values (To ensure consistency)
le=LabelEncoder()
data_df.Diabetic=le.fit_transform(data_df.Diabetic)
data_df.GENDER=le.fit_transform(data_df.GENDER)

In [12]:
#Splitting data into training and testing set
train_x, test_x, train_y, test_y=train_test_split(data_df.iloc[:,:6], data_df.Diabetic, test_size=0.3,random_state=43)

In [13]:
#Normalizing data using standard scaler
sc=StandardScaler()
train_x=sc.fit_transform(train_x)
test_x=sc.transform(test_x)

In [14]:
#Fetching diabetic and non diabetic records seperately from train set
train_x_diabetic=train_x[list(np.where(train_y==diabetic)[0])]
train_x_nondiabetic=train_x[list(np.where(train_y==nondia)[0])]

In [15]:
#Fetching diabetic and non diabetic records seperately from test set
test_x_diabetic=test_x[list(np.where(test_y==diabetic)[0])]
test_x_nondiabetic=test_x[list(np.where(test_y==nondia)[0])]

In [16]:
#Writing Functions for displaying evaluations
def evaluate(yt,yp):
    cf=confusion_matrix(yt, yp)
    acc=accuracy_score(yt, yp)
    return cf,acc

In [17]:
def display(yt,yp,model):
    cf,acc=evaluate(yt,yp)
    print('Model=',model,'\nConfusion matrix= ',cf,'\nAccuracy score= ',acc)

In [18]:
#Performing classification using linear SVM
lsvc=LinearSVC(random_state=0,C=10,max_iter=100000)
lsvc.fit(train_x, train_y)
train_yp=lsvc.predict(train_x)
test_yp=lsvc.predict(test_x)

In [19]:
#Display the result
display(train_y,train_yp,'Linear SVC: Validation')
display(test_y, test_yp,'Linear SVC: Testing')

Model= Linear SVC: Validation 
Confusion matrix=  [[232 137]
 [140 236]] 
Accuracy score=  0.6281879194630873
Model= Linear SVC: Testing 
Confusion matrix=  [[106  62]
 [ 61  91]] 
Accuracy score=  0.615625


In [20]:
#Coefficients of each feature in scaled x
print(lsvc.coef_)

[[ 0.03428299  0.30612897  0.4775003  -0.81866538  0.75757314 -0.04616436]]


In [21]:
#Intercept at scaled x
print(lsvc.intercept_)

[0.00939534]


In [22]:
#Rescaling the coefficients to original scale of the features of X
rescaled_coef=lsvc.coef_/np.sqrt(sc.var_)
print(rescaled_coef)

[[ 0.06862542  0.02514114  0.06038349 -0.06373233  0.15958324 -0.00584235]]


In [23]:
#The intercept in the original feature space
rescaled_intercept=rescaled_coef.dot(sc.mean_.T)+lsvc.intercept_
print(rescaled_intercept)

[10.64505955]


In [24]:
#Now identifying slacks or misclassified points in each class
non_dia_slacks=(lsvc.coef_.dot(train_x_diabetic.T)+lsvc.intercept_)
print(np.sum(non_dia_slacks<0))
dia_slacks=(lsvc.coef_.dot(train_x_nondiabetic.T)+lsvc.intercept_)
print(np.sum(dia_slacks>0))

140
137


In [25]:
#Creating custom dictionary for recall and precision
custom_scorer = {'recall':make_scorer(recall_score, pos_label=diabetic),
'precision':make_scorer(average_precision_score, pos_label=diabetic)}

In [26]:
#Tuning regularization parameter and retraining the model using best C value
gscv = GridSearchCV(LinearSVC(max_iter=int(1e7)), {'C':[1e-5,1e-4,1e-3,1e-2,1e-1,1,10,100,1000]},
cv=5,verbose=False,scoring=custom_scorer,refit='recall')
gscv.fit(train_x,train_y)
print(gscv.best_params_)

{'C': 0.001}


In [27]:
#Displaying the results based on C value 10 and best found value
display(train_y,train_yp,'For C=10: Training')
lsvc = LinearSVC(random_state=0,C=0.001,max_iter=100000)
lsvc.fit(train_x, train_y)
train_yp=lsvc.predict(train_x)
display(train_y,train_yp,'For C=0.001: Training')

Model= For C=10: Training 
Confusion matrix=  [[232 137]
 [140 236]] 
Accuracy score=  0.6281879194630873
Model= For C=0.001: Training 
Confusion matrix=  [[229 140]
 [133 243]] 
Accuracy score=  0.6335570469798658
